# Exploración del conjunto Credit Card Fraud Detection

Este notebook es descriptivo: no elimina registros ni ajusta escaladores o selectores. Las transformaciones que aprenden parámetros se ajustarán después de separar entrenamiento y prueba.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import seaborn as sns

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import TARGET_COLUMN
from src.data.load_data import diagnose_dataset, load_dataset

data = load_dataset()
diagnosis = diagnose_dataset(data)

## Estructura y calidad

Revisamos dimensiones, tipos, faltantes y duplicados antes de tomar cualquier decisión de limpieza. Si hubiera duplicados, se debe evaluar su distribución por `Class`; no se eliminarán automáticamente porque podría cambiar la representación de la clase fraude.

In [ ]:
print(f"Filas: {diagnosis['shape'][0]:,}; columnas: {diagnosis['shape'][1]}")
display(diagnosis['dtypes'].rename('tipo'))
display(diagnosis['missing_values'].rename('valores_faltantes'))
print(f"Registros duplicados exactos: {diagnosis['duplicate_rows']:,}")
display(diagnosis['duplicate_rows_by_class'].rename('duplicados_exactos_adicionales'))
display(diagnosis['descriptive_statistics'])

## Distribución de clases

La clase positiva (`Class = 1`) representa fraude. El fuerte desbalance vuelve engañosa la exactitud: un modelo que predijera siempre transacciones legítimas tendría una exactitud alta, pero sensibilidad nula para fraude.

In [ ]:
class_summary = diagnosis['class_counts'].to_frame('cantidad')
class_summary['porcentaje'] = diagnosis['class_percentages']
display(class_summary)

ax = sns.countplot(data=data, x=TARGET_COLUMN, hue=TARGET_COLUMN, legend=False, palette='Set2')
ax.set(title='Desbalance entre transacciones legítimas y fraudulentas', xlabel='Clase (0: legítima, 1: fraude)', ylabel='Número de transacciones')
plt.tight_layout()
plt.show()

## Distribuciones de `Amount` y `Time`

Estas visualizaciones son exploratorias. No se ajusta ningún escalador sobre el conjunto completo; para modelos sensibles a escala se utilizará un `Pipeline` ajustado únicamente con entrenamiento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=data, x='Amount', hue=TARGET_COLUMN, bins=60, stat='density', common_norm=False, ax=axes[0])
axes[0].set(title='Distribución de Amount por clase', xlabel='Monto', ylabel='Densidad')
sns.histplot(data=data, x='Time', hue=TARGET_COLUMN, bins=60, stat='density', common_norm=False, ax=axes[1])
axes[1].set(title='Distribución de Time por clase', xlabel='Tiempo desde la primera transacción', ylabel='Densidad')
plt.tight_layout()
plt.show()